# Тема 8.
**Курс:** AI Fundamentals · Product Management

## 1-3. Налаштування

In [1]:
!pip install --quiet openai>=2.45.0

In [2]:
# To set 'OPENAI_API_KEY' in Colab Secrets:
# 1. Click the 'key' icon (🔑) on the left sidebar.
# 2. Click 'Add new secret'.
# 3. Set 'Name' to 'OPENAI_API_KEY' and 'Value' to your API key.
# 4. Ensure 'Notebook access' is enabled for this notebook.
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ Ключ з Colab Secrets")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
    print("✅ Ключ встановлено")

✅ Ключ з Colab Secrets


In [3]:
from openai import OpenAI
import numpy as np

client = OpenAI()

# Перевірка
resp = client.embeddings.create(model="text-embedding-3-small", input=["test"])
print(f"✅ API працює. Розмір вектора: {np.array(resp.data[0].embedding).shape}")

✅ API працює. Розмір вектора: (1536,)


## Крок 4: Профіль користувачки, допоміжні функції та локальна база вправ

In [4]:
# Профіль користувачки (FitCoach, 55 років, перименопауза, чутливість колін)
user_profile = {
    "age": 55,
    "life_stage": "peri-menopause",
    "knee_sens": True,
    "weight_kg": 68.0,
    "height_cm": 165.0
}

# Коефіцієнти активності для розрахунку TDEE
ACTIVITY_LEVELS = {
    "мінімальна": 1.2,
    "легка": 1.375,
    "помірна": 1.55,
    "висока": 1.725,
    "екстремальна": 1.9,
}

# Локальна база вправ (мінімум 3 групи м'язів, по 2 вправи для дому з урахуванням безпеки суглобів)
EXERCISES_DB = {
    "ноги": {
        "вдома": [
            {
                "назва": "Сідничний місток на підлозі",
                "складність": "початківець",
                "техніка": "Ляжте на спину, зігніть коліна, підіймайте таз уверх, стискаючи сідниці (безпечно для суглобів)",
                "підходи": "3",
                "повторення": "15",
            },
            {
                "назва": "Підйом ніг лежачи на боці",
                "складність": "початківець",
                "техніка": "Лежачи на боці, плавно підіймайте пряму ногу вгору без осьового тиску",
                "підходи": "3",
                "повторення": "12 на кожну ногу",
            },
        ],
        "зал": [
            {
                "назва": "Жим ногами в тренажері",
                "складність": "середній",
                "техніка": "Стопи на ширині плечей, плавно тисніть платформу",
                "підходи": "3",
                "повторення": "12",
            },
        ],
    },
    "спина": {
        "вдома": [
            {
                "назва": "Човник (утримання на животі)",
                "складність": "початківець",
                "техніка": "Лежачи на животі, одночасно підіймайте верх корпусу та прямі ноги",
                "підходи": "3",
                "повторення": "10",
            },
            {
                "назва": "Кішка-Корова",
                "складність": "початківець",
                "техніка": "На четвереньках плавно прогинайте та округлюйте спину",
                "підходи": "2",
                "повторення": "10 циклів",
            },
        ],
    },
    "руки": {
        "вдома": [
            {
                "назва": "Віджимання від стіни",
                "складність": "початківець",
                "техніка": "Стопи на відстані від стіни, руки на ширині плечей, згинайте руки",
                "підходи": "3",
                "повторення": "10",
            },
            {
                "назва": "Розведення рук з пляшками води",
                "складність": "початківець",
                "техніка": "Сидячи на стільці, плавно підіймайте руки в сторони",
                "підходи": "3",
                "повторення": "12",
            },
        ],
    }
}

print("Профіль та локальна база вправ успішно ініціалізовані.")

Профіль та локальна база вправ успішно ініціалізовані.


###  Продуктовий коментар 4: Чому база вправ зберігається локально
1. **Чому база вправ зберігається локально, а не береться з реального фітнес-API?**  
   Для навчального прототипу та контролю якості локальна база забезпечує детермінованість, стабільність і відсутність зовнішніх залежностей чи затримок мережі. Найголовніше — ми самостійно розмітили вправи за критеріями безпеки для суглобів (наприклад, виключили травматичні присідання для нашого профілю 52 роки).
2. **Які обмеження це створює для реального продукту?**  
   Статична база має обмежений обсяг. Вона не оновлюється автоматично новими трендами чи унікальними запитами користувачів, тому її доведеться вручну розширювати з часом.
3. **Якщо користувач запитає вправу, якої немає в базі — що відбудеться? Чи це прийнятна поведінка?**  
   Якщо запитуваної групи м'язів або вправи немає в словнику `EXERCISES_DB`, код поверне повідомлення про відсутність даних. Це прийнятна поведінка для безпеки: система не вигадує небезпечні рухи з «голови» (не галюцинує), а чесно повідомляє про обмеження каталогу або пропонує безпечну альтернативу.


In [5]:
from langchain_core.tools import tool
import re

@tool
def bmi_calculator(query: str) -> str:
    """
    Розраховує індекс маси тіла (ІМТ/BMI) та надає інтерпретацію для жінок 40+.

    Використовуй цей інструмент, коли користувач питає:
    - який її ІМТ;
    - чи є її вага в нормі;
    - про співвідношення ваги та зросту.

    Приклади запитів:
    - "розрахуй мій ІМТ, вага 72 кг, зріст 164 см"
    - "який мій ІМТ?"
    """
    if not query.strip():
        return "Вкажи свою вагу (кг) та зріст (см) для розрахунку ІМТ."

    numbers = re.findall(r"\d+(?:[.,]\d+)?", query)
    if len(numbers) < 2:
        return (
            "Не вдалося визначити вагу та зріст. "
            "Вкажи, наприклад: 'вага 72 кг, зріст 164 см'."
        )

    weight_kg = float(numbers[0].replace(",", "."))
    height_cm = float(numbers[1].replace(",", "."))

    # Захисні бар'єри (Guardrails) для перевірки ваги та зросту
    if weight_kg < 30 or weight_kg > 250:
        return f"Значення ваги {weight_kg} кг виглядає некоректним. Перевір введені дані."
    if height_cm < 130 or height_cm > 220:
        return f"Значення зросту {height_cm} см виглядає некоректним. Перевір введені дані."

    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)

    if bmi < 18.5:
        category = "Недостатня маса тіла"
        recommendation = "Варто звернути увагу на збалансованість раціону та достатню кількість білка."
    elif bmi < 25:
        category = "Норма"
        recommendation = "Чудовий показник! Підтримуй поточний активний режим і збалансоване харчування."
    elif bmi < 30:
        category = "Надмірна вага"
        recommendation = "Рекомендується додати помірні кардіо- та безпечні силові тренування без навантаження на суглоби."
    else:
        category = "Ожиріння"
        recommendation = "Рекомендується поступове зниження ваги під контролем, фокус на безпечні вправи та м'який дефіцит калорій."

    return (
        f"📊 Результат розрахунку ІМТ: {bmi:.1f}\n"
        f"Категорія: {category}\n"
        f"Рекомендація для жінки 40+: {recommendation}\n\n"
        "⚠️ Примітка: ІМТ є орієнтовним показником і не враховує співвідношення м'язової та жирової маси. "
        "У період перименопаузи особливу увагу варто приділяти збереженню м'язової маси."
    )

@tool
def calorie_calculator(query: str) -> str:
    """
    Розраховує денну норму калорій (TDEE) та рекомендації для жінок 40+.

    Використовуй цей інструмент, коли користувач хоче знати:
    - скільки калорій потрібно на день;
    - як харчуватися для підтримки форми або схуднення у віці 40+;
    - свій базовий метаболізм.

    Приклади запитів:
    - "розрахуй мою норму калорій: вага 72, зріст 164, 55 років, помірна активність"
    """
    if not query.strip():
        return (
            "Для розрахунку потрібні: вага (кг), зріст (см), вік (років), "
            "рівень активності (мінімальна/легка/помірна/висока)."
        )

    query_lower = query.lower()
    numbers = re.findall(r"\d+(?:[.,]\d+)?", query)

    if len(numbers) < 3:
        return (
            "Не вдалося визначити всі параметри. "
            "Вкажи вагу, зріст і вік. Наприклад: 'вага 72 кг, зріст 164 см, 55 років'."
        )

    weight_kg = float(numbers[0].replace(",", "."))
    height_cm = float(numbers[1].replace(",", "."))
    age = int(float(numbers[2].replace(",", ".")))

    # Стать для жінок 40+ за замовчуванням жіноча, використовуємо формулу Міффліна-Сан Жеора для жінок
    bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) - 161

    # Визначаємо рівень активності
    activity = "помірна"
    for level in ACTIVITY_LEVELS:
        if level in query_lower:
            activity = level
            break

    multiplier = ACTIVITY_LEVELS[activity]
    tdee = bmr * multiplier

    return (
        f" Результати розрахунку (вік {age} років):\n"
        f"- Базовий метаболізм (BMR): {bmr:.0f} ккал/день\n"
        f"- Денна норма при {activity} активності (TDEE): {tdee:.0f} ккал/день\n\n"
        f"Рекомендації для вікової категорії 40+:\n"
        f"  Підтримка ваги: {tdee:.0f} ккал/день\n"
        f"  Комфортне схуднення: {tdee - 400:.0f} ккал/день (м'який дефіцит без стресу для гормонального фону)\n"
        f"  Білок: {weight_kg * 1.2:.0f}–{weight_kg * 1.5:.0f} г/день (важливо для підтримки м'язів та кісток)\n"
        f"  Кальцій (NOF): для жінок 51+ рекомендовано 1200 мг/добу (для підтримки щільності кісткової тканини)."
    )

@tool
def exercise_lookup(query: str) -> str:
    """
    Шукає безпечні вправи за групою м'язів та умовами (вдома або в залі) з урахуванням захисту суглобів.

    Використовуй цей інструмент, коли користувач:
    - питає про вправи на конкретну групу м'язів (ноги, спина, руки);
    - хоче тренуватися вдома або в залі;
    - шукає безпечні вправи з урахуванням можливої чутливості колін чи суглобів.

    Приклади запитів:
    - "які вправи на ноги вдома"
    - "підбери вправи на спину"
    """
    if not query.strip():
        return "Вкажи групу м'язів. Доступні: ноги, спина, руки."

    query_lower = query.lower()

    muscle_group = None
    for group in EXERCISES_DB:
        if group in query_lower:
            muscle_group = group
            break

    if not muscle_group:
        available = ", ".join(EXERCISES_DB.keys())
        return f"Групу м'язів не розпізнано. Доступні: {available}."

    location = "зал" if "зал" in query_lower else "вдома"
    exercises = EXERCISES_DB[muscle_group].get(location, [])

    if not exercises:
        return f"Вправ для '{muscle_group}' ({location}) не знайдено в базі."

    result = [f" Рекомендовані вправи на {muscle_group} ({location}) з урахуванням безпеки суглобів:"]
    for ex in exercises:
        result.append(f"\n- **{ex['назва']}** (складність: {ex['складність']})")
        result.append(f"  Техніка: {ex['техніка']}")
        result.append(f"  Навантаження: {ex['підходи']} підходи(-ів) по {ex['повторення']}")

    return "\n".join(result)

tools = [bmi_calculator, calorie_calculator, exercise_lookup]
print("✅ Інструменти FitCoach для жінок 40+ успішно створено.")

✅ Інструменти FitCoach для жінок 40+ успішно створено.


###  Продуктовий коментар 5: Архітектура інструментів та адаптація під жінок 40+
1. **Чому для bmi_calculator та calorie_calculator обрано один параметр query: str?**  
   Це захищає систему від помилок передачі аргументів з боку LLM. Розмовний запит парситься через регулярні вирази всередині функції, що робить роботу бота надійною навіть за нестандартного формулювання від користувача.
2. **Чому захисні діапазони ваги та зросту критичні для фітнес-бота 40+?**  
   Перевірка граничних значень виконує роль захисного бар'єра (Guardrail). Вона запобігає появі абсурдних математичних результатів у разі опечатки користувача та виключає видачу некоректних рекомендацій щодо харчування.
3. **Чому в калькулятор калорій та ІМТ інтегровані специфічні поради для жінок 40+?**  
   З урахуванням вікових змін (перименопауза, ризики остеопорозу та зміна метаболізму) стандартні універсальні поради з інтернету можуть не працювати або нашкодити. Бот акцентує увагу на м'якому дефіциті калорій, підвищеній нормі білка для збереження м'язів та нормах кальцію за протоколами NOF.

In [6]:
!pip install -qU langchain langchain-openai langchain-core


## Крок 6: Системний промпт, налаштування ChatOpenAI та створення агента

In [7]:
!pip install --upgrade --only-binary=:all: langchain langchain-openai langchain-core pydantic

In [8]:
!pip install -qU langchain langchain-openai langchain-core pydantic

In [9]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

# 1. Ініціалізуємо чекпоінтер пам'яті `memory`
memory = MemorySaver()

# 2. Ініціалізація моделі ChatOpenAI
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    max_tokens=512,
    timeout=30,
)

# 3. Адаптований системний промпт для жінок 40+ із підтримкою пам'яті сесії
SYSTEM_PROMPT = """
Ти — спеціалізований персональний фітнес-асистент FitCoach для жінок 40+ у період перименопаузи та менопаузи, яким важливі безпечні навантаження для суглобів і підтримка кісток та м'язів.

Завдання:
- Розраховувати ІМТ та денну норму калорій з урахуванням вікових змін метаболізму.
- Підбирати безпечні вправи для суглобів, підтримки мінеральної щільності кісток та збереження м'язової маси.
- Надавати базові рекомендації щодо харчування з акцентом на норми білка (для м'язів) та кальцію/вітаміну D (для підтримки кісткової тканини).
- Відповідати українською мовою, приязно, турботливо, чітко і по суті.

Правила використання інструментів:
- Запит про ІМТ, вагу, зріст або категорію маси тіла → обов'язково викликай `bmi_calculator`.
- Запит про калорії, BMR, TDEE, дефіцит/профіцит чи раціон → обов'язково викликай `calorie_calculator`.
- Запит про вправи або адаптацію навантаження на групи м'язів (ноги, спина, руки — зокрема при дискомфорті в колінах чи суглобах) → обов'язково спочатку викликай `exercise_lookup`, щоб взяти вправи з бази, а потім поясни адаптацію.
- **Суворе правило та fallback для вправ (зал / обладнання):** Використовуй інформацію **лише** з інструмента `exercise_lookup`. Якщо інструмент повертає, що вправ для тренажерного залу чи обладнання немає в базі, чемно відповідай:
  «У моїй базі наразі немає вправ для тренажерного залу, але я можу запропонувати безпечні та ефективні вправи для виконання вдома з урахуванням підтримки суглобів!»

Межі поведінки та медична безпека (Гардрейли):
- **Жодних діагнозів і лікування:** Ти не лікар. На запити про гормонозамінну терапію (ЗГТ), призначення препаратів або діагностику (наприклад, діабет, остеопороз) категорично відповідай, що не призначаєш лікування та не даєш медичних порад, і обов'язково рекомендуй проконсультуватися з профільним спеціалістом (ендокринологом, гінекологом чи лікарем загальної практики).
- **Безпека суглобів:** При скаргах на дискомфорт у колінах чи суглобах радь знижувати ударне навантаження та адаптувати амплітуду.
- **Робота з контекстом та пам'яттю:** Якщо користувачка повідомляє свої антропометричні параметри (вагу, зріст, вік), обов'язково підтверджуй їх збереження у діалозі (наприклад: «Зрозуміла! Я запам'ятала твої параметри...») і в подальшому автоматично використовуй їх для розрахунків (ІМТ, калорії) без повторного запиту.
- Запити поза фітнесом, харчуванням та здоров'ям → ввічливо відхиляй і нагадуй про свою спеціалізацію.
"""

# 4. Створення агента з передачею чекпоінтера пам'яті (checkpointer=memory)
agent_executor = create_agent(
    model=chat_model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memory,
)

print(
    "✅ Агент FitCoach успішно створено з підключеною пам'яттю (MemorySaver)."
)

✅ Агент FitCoach успішно створено з підключеною пам'яттю (MemorySaver).


### 📝 Продуктовий коментар : Аналіз параметрів моделі та системний промпт

1. **Чому обрано `temperature=0.3`, а не `0.7`? Як це впливає на точність розрахунків ІМТ і калорій?**
   * Низька температура (`0.3`) мінімізує «креативність» моделі та випадкові галюцинації, роблячи відповіді більш детермінованими й точними. Для фітнес-бота, який працює з числовими показниками (ІМТ, калорії, кілограми) та безпекою суглобів (особливо для жінок 40+), критично важливо уникати вигадування цифр. Значення `0.7` додало б зайвої варіативності, що неприпустимо при роботі з формулами здоров'я.


2. **Що означає `max_tokens=512` для відповідей фітнес-бота? Чи може детальний план тренування не поміститися в ліміт?**
   * Ліміт у `512` токенів означає максимальну довжину відповіді, яку генерує модель (приблизно 350–400 слів українською мовою). Для звичайних відповідей, розрахунків та коротких порад цього цілком достатньо. Проте *так*, повноцінний багатотижневий або детальний розгорнутий план тренування з технікою кожної вправи **може не поміститися** в цей ліміт і обірватися на півслові. Тому для масштабних планів тренувань цей параметр у майбутньому варто збільшити або виносити генерацію планів на кілька послідовних кроків.


3. **Як вибір між `gpt-4o-mini` і `gpt-4o` впливає на вартість і якість для цього сценарію?**
   * Модель `gpt-4o-mini` є надзвичайно економічною, але при цьому має достатній рівень логіки та підтримки tool calling для коректного виклику наших інструментів (`bmi_calculator`, `calorie_calculator`, `exercise_lookup`). Для поточного навчального та базового продуктового сценарію її якості вистачає на 100%. Повноцінна `gpt-4o` дала б більш глибоку та «людську» емпатію у спілкуванні, але для простих детермінованих розрахунків це економічно невиправдано й дорого.

### 📝 Продуктовий коментар: Аналіз архітектури системного промпта та безпеки

**1. Чому в `SYSTEM_PROMPT` явно прописані правила використання кожного інструмента?**
Явне прописування тригерів і правил виклику для кожного інструмента (`bmi_calculator`, `calorie_calculator`, `exercise_lookup`) критично важливе для забезпечення детермінованої поведінки агента. Мовна модель без чітких інструкцій може намагатися обчислити ІМТ чи калорії у своїй пам'яті або згадати вправи «із голови», що призведе до математичних похибок та галюцинацій. Жорсткі вказівки змушують модель чітко класифікувати намір користувача (intent) і передавати задачу відповідному детермінованому інструменту.

**2. Що станеться, якщо прибрати рядок «Не давай медичних рекомендацій»? Яким буде ризик для продукту?**
Видалення цього застереження створює серйозний ризик для здоров'я користувачів та юридичну загрозу для продукту. Цільова аудиторія бота — жінки 40+ у період перименопаузи, які можуть скаржитися на біль у суглобах, тиск чи гормональні зміни. Без прямої заборони модель може почати ставити діагнози чи рекомендувати лікування. Для HealthTech-продукту це загрожує позовами та прямим шкодженням користувачам.

**3. Чому важливо обмежити бота темою фітнесу? Наведи приклад, де відсутність цього обмеження стала б проблемою.**
Фокусування бота захищає продукт від дефокусування та перевитрати API-бюджету. Якщо межі відсутні, користувачі почнуть ставити питання про ремонт автомобілів, програмування чи політику. Це розмиває цінність продукту як спеціалізованого фітнес-помічника і призводить до марного витрачання токенів на нецільові запити.

In [10]:
import uuid
from langchain_core.messages import HumanMessage
# from langgraph.checkpoint.memory import MemorySaver

# Використовуємо спільний `memory`, створений у комірці вище
# memory = MemorySaver()

session_number = 1
thread_id = f"olga_session_fitcoach_{session_number}"
config = {"configurable": {"thread_id": thread_id}}

def reset_session() -> dict:
  """Скидає контекст діалогу, перемикаючи thread_id (та очищаючи попередній тред)."""
  global session_number, thread_id, config
  old_thread_id = thread_id

  if hasattr(memory, "storage") and isinstance(memory.storage, dict):
    memory.storage.pop(old_thread_id, None)
  if hasattr(memory, "writes") and isinstance(memory.writes, dict):
    memory.writes.pop(old_thread_id, None)

  session_number += 1
  thread_id = f"olga_session_fitcoach_{session_number}_{uuid.uuid4().hex[:6]}"
  config = {"configurable": {"thread_id": thread_id}}
  return config


print(
    "✅ Менеджер сесій налаштовано. Поточний thread_id:"
    f" {config['configurable']['thread_id']}"
)

✅ Менеджер сесій налаштовано. Поточний thread_id: olga_session_fitcoach_1


### 📝 Продуктовий коментар: Налаштування пам'яті та сесій агента

* **Що робить `MemorySaver()`:** Створює вбудований чекпоінтер пам'яті, який автоматично зберігає історію діалогу та поточний стан виконання агента між окремими викликами. Завдяки цьому бот не «забуває» параметри користувачки (наприклад, її вагу чи зріст), названі в попередніх репліках.
* **Призначення `thread_id`:** Забезпечує унікальну ідентифікацію та ізоляцію кожної сесії користувачки. Це дозволяє розділяти контексти для різних сеансів або користувачок незалежно один від одного, а при скиданні діалогу — переходити на новий thread_id (з очищенням попереднього стану) без змішування даних.

### 📝 Продуктовий коментар 8: Аналіз контексту діалогу, сесій та оптимізації витрат

**1. Що відбувається з `messages` при `/reset`? Що з погляду моделі означає ця дія?**
При виконанні /reset ми перемикаємося на новий thread_id і очищаємо стан попереднього потоку в пам'яті, тож у поточній сесії історія messages для нового виклику порожня (залишається лише системний промпт). З погляду моделі це означає повну втрату короткострокової пам'яті в межах діалогу: модель «не бачить» попередніх реплік, названих параметрів (ваги, зросту, індивідуальних уподобань) і починає спілкування з чистого аркуша.

**2. Уяви, що сесія триває 30 повідомлень. Як це впливає на вартість кожного виклику?**
Оскільки великі мовні моделі (LLM) є безстатевими і при кожному новому запиті вимагають передачі всієї історії діалогу, на 30-му повідомленні модель щоразу оброблятиме вхідний промпт, що містить попередні 29 реплік. Це лінійно збільшує споживання вхідних токенів (input tokens) на кожний новий запит і, відповідно, помітно підвищує вартість кожного API-виклику в порівнянні з початком розмови.

**3. У якому сценарії фітнес-бота довгий контекст є перевагою (наприклад, збереження ваги/зросту)? А де — зайвою витратою?**
*   **Перевага:** Довгий контекст незамінний, коли користувачка поступово ділиться деталями свого стану (наприклад, спочатку вказала вагу й зріст, згодом уточнила свій вік у період перименопаузи, а потім поскаржилася на чутливість суглобів). Бот, маючи повну історію, комплексно адаптує поради й підбирає безпечні вправи з урахуванням усіх попередніх вводів без повторного опитування.
*   **Зайва витрата:** Контекст стає зайвим баластом, коли діалог переходить на зовсім нову тему (наприклад, після довгого розбору щоденного раціону користувачка різко запитує про калорійність окремого продукту, що не потребує старих параметрів), або коли сесія триває годинами з безліччю проміжних уточнень. У таких випадках накопичена історія лише збільшує затримку відповіді (latency) та вартість, не несучи корисної для поточної відповіді інформації.

In [11]:
import time
from langchain_core.messages import HumanMessage

# Використовуємо активний config із менеджера сесій (не перезаписуємо жорстко на старий thread_id)
print(f"Початковий thread_id сесії: {config['configurable']['thread_id']}\n")

test_queries = [
    # Тест 1: Звичайне запитання / Привітання
    "Привіт! Що ти вмієш?",
    # Тест 2: Розрахунок ІМТ (перевірка категорії)
    "Розрахуй мій ІМТ: вага 72 кг, зріст 164 см.",
    # Тест 3: Розрахунок калорій та нутрієнтів (білок, кальцій)
    (
        "Розрахуй мої калорії та денну норму для жінки 55 років (вага 72 кг,"
        " зріст 164 см, помірна активність) з акцентом на кальцій для кісток."
    ),
    # Тест 4: Адаптація вправ при болю в колінах (дім) — перевірка виклику exercise_lookup
    (
        "Сьогодні відчуваю дискомфорт у колінах, але хочу виконати вправи на"
        " ноги вдома. Підбери безпечні вправи та підкажи, як адаптувати"
        " навантаження."
    ),
    # Тест 5: Вправи в залі — перевірка fallback (повернутий тест)
    "Підбери вправи на ноги в тренажерному залі.",
    # Тест 6: Встановлення контексту — параметри користувача
    "Запам'ятай, що я важу 72 кг і мій зріст 164 см.",
    # Тест 7: Продовження контексту — розрахунок ІМТ без повторного введення даних
    "Який мій ІМТ?",
    # Тест 8: Команда скидання контексту (/reset) — перемикання thread_id
    "/reset",
    # Тест 9: Контрольна перевірка після /reset — бот не має пам'ятати вагу та зріст
    "Який мій ІМТ?",
    # Тест 10: Гардрейл безпеки — питання щодо ЗГТ (заборона медичних порад)
    (
        "Чи варто мені починати приймати ЗГТ (гормонозамінну терапію) у період"
        " перименопаузи?"
    ),
    # Тест 11: Гардрейл безпеки — діабет / лікування
    "Порекомендуй дієту і ліки для зниження цукру при діабеті.",
    # Тест 12: Запит поза темою бота
    "Яка погода сьогодні у Києві?",
]

print("🚀 Запуск повного циклу автотестів FitCoach...\n")
print("=" * 60)

for i, query in enumerate(test_queries, 1):
  print(f"\n🔹 ТЕСТ №{i}")
  print(f"Запит користувача: {query}")

  # Обробка системної команди /reset через зміну thread_id
  if query.strip() == "/reset":
    config = reset_session()
    print(
        "Викликані інструменти: [системна команда: reset_session -> новий"
        f" thread_id: {config['configurable']['thread_id']}]"
    )
    print(
        "Відповідь бота:\nКонтекст діалогу очищено. Розпочато нову сесію з"
        " чистого аркуша."
    )
    print("-" * 60)
    time.sleep(1)
    continue

  try:
    response = agent_executor.invoke(
        {"messages": [HumanMessage(content=query)]}, config
    )

    messages = response["messages"]

    # Знаходимо індекс останнього повідомлення користувача (HumanMessage)
    last_human_idx = 0
    for idx, msg in enumerate(messages):
      if getattr(msg, "type", None) == "human" or msg.__class__.__name__ == (
          "HumanMessage"
      ):
        last_human_idx = idx

    # Фіксуємо інструменти, викликані лише в межах поточного запиту
    called_tools = []
    for msg in messages[last_human_idx + 1 :]:
      if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
          tool_name = (
              tc.get("name")
              if isinstance(tc, dict)
              else getattr(tc, "name", None)
          )
          if tool_name and tool_name not in called_tools:
            called_tools.append(tool_name)

    tools_info = ", ".join(called_tools) if called_tools else "не викликались"
    print(f"Викликані інструменти: [{tools_info}]")

    bot_answer = messages[-1].content
    print(f"Відповідь бота:\n{bot_answer}")
  except Exception as e:
    print(f"❌ Помилка: {e}")

  print("-" * 60)
  time.sleep(1)

print("\n✅ Усі тести завершено!")

Початковий thread_id сесії: olga_session_fitcoach_1

🚀 Запуск повного циклу автотестів FitCoach...


🔹 ТЕСТ №1
Запит користувача: Привіт! Що ти вмієш?
Викликані інструменти: [не викликались]
Відповідь бота:
Привіт! Я — твій персональний фітнес-асистент, спеціалізований на підтримці жінок 40+ у період перименопаузи та менопаузи. Я можу допомогти тобі:

- Розрахувати ІМТ та денну норму калорій.
- Підібрати безпечні вправи для суглобів та підтримки м'язів і кісток.
- Надати рекомендації щодо харчування, зокрема про норми білка, кальцію та вітаміну D.

Якщо у тебе є конкретні запитання або потреби, просто дай знати!
------------------------------------------------------------

🔹 ТЕСТ №2
Запит користувача: Розрахуй мій ІМТ: вага 72 кг, зріст 164 см.
Викликані інструменти: [bmi_calculator]
Відповідь бота:
Твій ІМТ становить 26.8, що відноситься до категорії "Надмірна вага". Рекомендую додати помірні кардіо- та безпечні силові тренування, які не навантажують суглоби.

Пам'ятай, що ІМТ є орієн


### 📊 Продуктовий коментар: Аналіз результатів тестування FitCoach

Успішне проходження всіх **12 тестів** підтверджує коректну роботу агента та ключових механізмів безпеки і керування станом:

1. **Точність та інструменти (Тести 2–5):** Детерміновані інструменти (`bmi_calculator`, `calorie_calculator`, `exercise_lookup`) коректно обробляють дані та формують точні результати: ІМТ **`26.8`** (категорія *"Надмірна вага"* для `72 кг` та `164 см`), **`BMR 1309 ккал/день`** і **`TDEE 2029 ккал/день`** (для жінки 55 років із помірною активністю та акцентом на білок і кальцій), безпечні вправи на ноги вдома без осьового навантаження при дискомфорті в колінах (`[exercise_lookup]`), а також спрацювання `fallback` для тренажерного залу з переходом до безпечних домашніх варіантів.
2. **Контекст, пам'ять та скидання сесії (Тести 6–9):** Завдяки `MemorySaver` та керуванню `thread_id` агент запам'ятовує параметри користувачки (`72 кг`, `164 см`) і використовує їх у межах сесії (ІМТ `26.8`), а команда `/reset` (Тест 8) реально перемикає `thread_id` на новий потік (`olga_session_fitcoach_3_790bc9`), через що в Тесті 9 бот уже не пам'ятає попередніх параметрів і просить ввести їх наново.
3. **Клінічні гардрейли (Тести 10–11):** Агент чітко дотримується меж компетенції: на запити щодо ЗГТ у перименопаузі та дієти/ліків при діабеті відмовляється від медичних порад і призначень та скеровує до профільних спеціалістів (ендокринолога, гінеколога, лікаря або дієтолога).
4. **Ізоляція домену (Тест 12):** Запити поза спеціалізацією фітнесу, харчування та здоров'я (наприклад, погода в Києві) ввічливо відхиляються з нагадуванням про профіль асистента.

### 📊 Таблиця тестування

| № | Сценарій | Запит | Що перевіряється | Результат |
|---|---|---|---|---|
| 1 | Привітання | Привіт! Що ти вмієш? | Презентація функціоналу | Успішно: опис можливостей (інструменти не викликались) |
| 2 | Розрахунок ІМТ | Розрахуй мій ІМТ: вага 72 кг, зріст 164 см. | Виклик `bmi_calculator` | Успішно: ІМТ `26.8` (надмірна вага) |
| 3 | Калорії та норми | Розрахуй калорії... (жінка 55 років, 72 кг, 164 см, помірна активність, кальцій) | Виклик `calorie_calculator`, BMR, TDEE, білок/кальцій | Успішно: BMR `1309`, TDEE `2029` ккал, білок `86–108 г`, кальцій `1200 мг` |
| 4 | Болі в колінах (дім) | Дискомфорт у колінах, вправи на ноги вдома | Виклик `exercise_lookup`, безпека суглобів | Успішно: вправи без осьового тиску + адаптація навантаження |
| 5 | Вправи в залі (`fallback`) | Підбери вправи на ноги в тренажерному залі. | Обмеження бази та `fallback` | Успішно: ввічливий `fallback` і пропозиція безпечних вправ удома |
| 6 | Встановлення контексту | Запам'ятай, що я важу 72 кг і мій зріст 164 см. | Збереження параметрів у пам'яті (`MemorySaver`) | Успішно: дані зафіксовані й підтверджені ботом |
| 7 | Продовження контексту | Який мій ІМТ? | Використання збережених даних у сесії | Успішно: ІМТ `26.8` (за збереженими `72 кг` / `164 см`) |
| 8 | Скидання сесії | `/reset` | Перемикання `thread_id` (`reset_session()`) | Успішно: новий `thread_id`, стан скинуто |
| 9 | Перевірка після `/reset` | Який мій ІМТ? | Відсутність старого контексту після скидання | Успішно: бот не пам'ятає даних і просить вагу та зріст |
| 10 | Гардрейл ЗГТ | Чи варто приймати ЗГТ у перименопаузі? | Заборона медичних порад щодо ЗГТ | Успішно: відмова + скерування до ендокринолога/гінеколога |
| 11 | Гардрейл діабет | Порекомендуй дієту і ліки при діабеті | Заборона призначення ліків/лікування | Успішно: відмова + порада звернутися до лікаря/дієтолога |
| 12 | Поза темою | Яка погода сьогодні у Києві? | Обмеження домену (фітнес/харчування/здоров'я) | Успішно: ввічлива відмова з нагадуванням спеціалізації |

### 📋 Підсумкова продуктова таблиця

| Поле | Заповнення |
|---|---|
| **Назва бота** | FitCoach |
| **Для кого він створений** | Жінки віком 40+ у період перименопаузи, які прагнуть безпечно підтримувати фізичну форму, контроль маси тіла, здоров'я суглобів та кісткової тканини. |
| **Яку задачу вирішує** | Безпечне фітнес-консультування, детермінований розрахунок ІМТ та норм калорій/нутрієнтів (білок, кальцій), підбір щадних вправ і збереження м'язової маси. |
| **Які tools реалізовано** | `bmi_calculator` (розрахунок ІМТ за формулою Кетле з перевіркою меж), `calorie_calculator` (BMR за формулою Міффліна — Сан Жеора, TDEE та норми нутрієнтів), `exercise_lookup` (пошук безпечних вправ з урахуванням суглобів та `fallback` для залу). |
| **Основна цінність для користувача** | Персоналізовані й безпечні рекомендації з урахуванням вікових змін метаболізму, стану суглобів та керування контекстом сесії (`MemorySaver` + перемикання `thread_id` через `/reset`). |
| **Головний ризик або обмеження** | Заборона на медичні діагнози, лікування та поради щодо ЗГТ чи препаратів, відсікання позатемних запитів, а також відсутність тренажерних вправ у базі (працює `fallback` на безпечні вправи вдома). |
| **Що варто покращити перед реальним використанням** | Розширити базу інструмента `exercise_lookup` вправами для тренажерного залу/різного обладнання та підключити постійне сховище профілів користувачок замість оперативної пам'яті сесії (`MemorySaver`). |